# 주제 : 인공위성 영상 데이터로 산림 유형을 분류하는 AI 모델 개발

LCA 산정에 필요한 산림 데이터로 활용 가능한지 평가
LCA 기준에서 탄소축적량이 쓰일 수 있다고 생각하고 아래 실험을 진행

> **요약** : 분류 성능(F1)은 나오지만, LCA 산정에 필요한(?) 탄소량이 오차가 큼

## 1. 방법

위성 영상으로 산림 유형을 분류함 (침엽수림 / 활엽수림 / 혼효림)
분류 결과에 탄소계수를 곱해서 탄소축적량을 구함.

## 2. 모델 만든 과정

정답 데이터 : 1:5000 임상도. 산림청이 제공함
입력 : Google Earth Engine 에서 Sentinel-2 위성 10m 해상도, 무료
임상도 경계를 그대로 위성 촬영 범위로 설정.

**모델 설정**

| | |
|---|---|
| feature 38개 | 여름 10 + 낙엽 10 + 식생지수 + 주변 픽셀 |
| 분류기 | RandomForest (딥러닝 아님. 가져옴) |
| 학습량 | 모든 조건에서 20만 픽셀 |
| 라벨 | 조사 갱신년도 2024 이후만 사용 (2020 이후도 쓸 수 있지만 2024로 통일) |

## 3. 학습 결과

대전 지역으로 학습했는데 F1 0.628 탄소오차 +0.1% 큰 문제 없음

## 4. 타일 분할

픽셀을 랜덤하게 나누면 학습 평가 타일 인접 가능성 발생 →
500픽셀 타일 단위로 묶음
타일도 동일한 문제 발생할 수 있음 → 동서 분리

결과 : F1 0.572 탄소 오차 +0.77%
같은 지역, 같은 데이터, 같은 모델인데 평가 영역 분리되니 오차가 10배 이상
→ 지역 바뀌면??

## 5. 지역 변경 : 대전 홍천 순천

| 학습 → 평가 | F1 | 탄소 오차 |
|---|---|---|
| 대전 → 대전 | 0.624 | −1.05% |
| 대전 → 홍천 | 0.569 | +2.73% |
| 대전 → 순천 | 0.550 | −1.57% |
| 홍천 → 홍천 | 0.625 | +0.12% |
| 홍천 → 대전 | 0.581 | −1.51% |
| 홍천 → 순천 | 0.539 | −3.48% |
| 순천 → 순천 | 0.630 | +0.84% |
| 순천 → 대전 | 0.583 | +3.12% |
| 순천 → 홍천 | 0.484 | +5.92% |

## 6. F1보다 탄소량 오차가 큼

F1은 0.626 → 0.551
잘못 분류한 픽셀도 1.25배
탄소량은 0.12% → 5.92%

F1만으로 탄소오차 측정 어려움 (오차의 부호도 다름)
총량은 상쇄된 값임. 타일간 표준편차까지 고려해야함
(표준편차는 있지만 어느정도까지가 유효한 값인지 모름)

## 7. 원인 찾기

지역마다 기준이 다름

원인 1 : 생략

원인 2 : 지역 offset

| 지역 | 활엽 | 침엽 | 두 클래스의 경계 |
|---|---|---|---|
| 홍천 | 0.369 | 0.538 | 0.453 |
| 대전 | 0.440 | 0.655 | 0.548 |
| 순천 | 0.515 | 0.742 | 0.629 |

순천의 활엽수(0.515)가 홍천의 침엽수(0.538)와 거의 동일

지역 간 차이(0.204)가 클래스 간 차이(0.169)보다 큼
즉 "활엽과 침엽"보다 "어느 지역이냐"가 더 중요

## 8. 고치려고 해본 것

| | |
|---|---|
| 1. 모델을 키워봄 | 효과 적음 |
| 2. 수종 추가 | 적음 |
| 3. 지역별 offset 정규화 | 반응은 큰데 과보정. 6조합 전부 부호가 반대로 뒤집힘 |


지역마다 자기 중앙값을 빼서 기준선을 맞춤 (위성 통계만 사용, 라벨 불필요)
방향은 맞았으나 세기가 과함 → 0을 지나 반대편으로 넘어감

## 9. 지금까지 모르는 것

1. LCA 기준
2. 오차범위

## 10. 해보고 있는 것

1. 지역 늘려서 보정계수 alpha 찾기

8-3이 지나쳤으므로 세기를 조절하는 계수 alpha 도입 (0 = 안 함, 1 = 전량)
6조합 전부 부호가 바뀌었다는 건 0과 1 사이에 교차점이 있다는 뜻
alpha는 라벨 있는 지역들로 한 번 찾고, 적용할 때는 위성영상만 있으면 됨